# CBAM Corridor Cost Model
### Halifax–Hamburg and Ningbo–Felixstowe, hydrogen and ammonia

Two corridors sit under different carbon regimes in the same calendar year, and that asymmetry is what
the study is about:

- **Halifax → Hamburg** falls under EU CBAM (live since 1 January 2026), EU ETS Maritime, and FuelEU Maritime.
- **Ningbo → Felixstowe** falls under UK CBAM, which does not begin until 1 January 2027, and UK ETS Maritime,
  which covers only time spent physically in a UK port. The ocean crossing carries no UK carbon liability at all.

Every maritime figure here comes from Gayu's notebooks. Nothing has been substituted, rounded differently,
or filled in. Section 1 checks that by reproducing her published outputs before anything is built on top.

The model works in two layers. Gayu's maritime costs are per voyage; CBAM liability is per tonne of
product, because embedded emissions are expressed per tonne. Her cargo capacity notebook supplies the
tonnage that joins them, so **total carbon compliance cost per tonne is now computable**.

Full delivered cost is not, because production, conversion and freight cost still have no owner.

In [1]:
import warnings
import pandas as pd

from cbam_model import data_io, runner
from cbam_model.analysis import outputs, sensitivity
from cbam_model.config import regulatory_constants as rc
from cbam_model.config import vessel_logistics as vl
from cbam_model.validation import gayu_reproduction, reference_case

pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', 200)

## 1. Reproducing Gayu's notebooks

Distances, voyage time, fuel burn, CO2, and the EU ETS, UK ETS and FuelEU costs that follow. If any of
these stop matching, a shared assumption has moved and nothing downstream should be trusted until it is
understood.

In [2]:
print(gayu_reproduction.format_report())

Reproduction of Gayu's maritime notebooks

  Gas carrier (VLGC/VLAC), primary case
    [ok  ] Halifax-Hamburg distance (nm)          modelled     2,962.00  Gayu     2,962.00
    [ok  ] Ningbo-Felixstowe Suez distance (nm)   modelled    10,403.00  Gayu    10,403.00
    [ok  ] Ningbo-Felixstowe Cape distance (nm)   modelled    14,815.00  Gayu    14,815.00
    [ok  ] Daily fuel burn (t/day)                modelled        41.30  Gayu        41.30
    [ok  ] HH voyage days                         modelled         8.30  Gayu         8.30
    [ok  ] HH voyage fuel (t)                     modelled       342.80  Gayu       342.80
    [ok  ] HH voyage CO2 (t)                      modelled     1,080.20  Gayu     1,080.20
    [ok  ] HH voyage CO2e (t)                     modelled     1,097.00  Gayu     1,097.00
    [ok  ] NF voyage days                         modelled        29.30  Gayu        29.30
    [ok  ] NF voyage fuel (t)                     modelled     1,210.10  Gayu     1,210.10
    [ok

## 2. What comes from where

Gayu's figures supersede the original build spec wherever the two disagree, and in one place the spec was
badly wrong.

| quantity | build spec | Gayu | effect |
|---|---|---|---|
| Halifax–Hamburg distance | ~6,300 nm | **2,962 nm** | spec overstated Atlantic voyage emissions by >2x |
| Ningbo–Felixstowe (Suez) | ~11,200 nm | 10,403 nm | minor |
| Cape of Good Hope diversion | +3,500 nm | +4,412 nm | minor |
| VLSFO carbon factor | 3.114 tCO2/t | 3.151 tCO2/t | IMO MEPC 82/6/38 |
| EU ETS 2026 price | 65 / 82 / 126 | **70 / 80 / 90** | the spec's figures were a 2030 forecast |

Her FuelEU calculation divides the compliance deficit by the actual GHG intensity, which is what Annex IV
Part B requires. The build spec omitted that divisor and would have overstated the penalty by roughly
ninety times.

In [3]:
emissions, logistics, commercial = data_io.load_inputs()
logistics[['corridor', 'vessel_class', 'route_scenario', 'distance_nm', 'voyage_days',
           'voyage_co2_t', 'port_in_port_emissions_t']]

,corridor,vessel_class,route_scenario,distance_nm,voyage_days,voyage_co2_t,port_in_port_emissions_t
0,halifax_hamburg,VLGC/VLAC,suez,2962.0,8.3,1080.2,58.51
1,ningbo_felixstowe,VLGC/VLAC,suez,10403.0,29.3,3813.0,58.51
2,ningbo_felixstowe,VLGC/VLAC,cape,14815.0,41.7,5426.7,58.51


## 3. Carbon prices

The two regimes are priced from different sources and on different bases, so they are never combined.

**EU ETS.** Two anchor years. 2026 is ESMA's near-term market range, from Gayu. 2030 is the consensus
forecast aggregating Bloomberg, ABN Amro, Refinitiv, ICIS, S&P Global, Aurora and the Potsdam Institute.
Applying a near-term price to 2030, or a 2030 forecast to 2026, would both be wrong, so the model
interpolates between them.

**UK ETS.** £49.41/tCO2e, the UK ETS Authority determination for the scheme year beginning 1 January 2026.
It is formally the civil-penalty price, but it is calculated from twelve months of 2026 UKA December
futures settlement prices, so it is market-derived rather than administrative. The £40 and £60 figures
bracket it as a sensitivity range and are not forecasts.

Headline tables keep the two currencies apart, as Gayu's notebooks do. A GBP/EUR rate is now sourced
(ECB, 23 July 2026) and is applied only where a single-currency comparison is explicitly labelled.

The UK price has two variants. `frozen` holds the sourced 2026 official determination flat across
every year, which is the baseline and is conservative rather than correct: only one year was ever
sourced. `linked` runs the EU-UK ETS linkage scenario, under which the UK price converges on the EU
price by 2029. Linkage is NOT law, so it may only ever appear as a labelled scenario.

In [4]:
pd.DataFrame([
    {'year': y, 'scenario': s,
     'eu_ets_eur_per_tco2e': rc.eu_ets_price(y, s),
     'uk_ets_gbp_frozen': rc.uk_ets_price(y, s, 'frozen'),
     'uk_ets_gbp_linked': rc.uk_ets_price(y, s, 'linked')}
    for y in rc.CBAM_FACTOR if y <= 2030 for s in rc.PRICE_SCENARIOS
])

,year,scenario,eu_ets_eur_per_tco2e,uk_ets_gbp_frozen,uk_ets_gbp_linked
0,2026,low,70.00,40.00,40.000000
1,2026,medium,80.00,49.41,49.410000
2,2026,high,90.00,60.00,60.000000
3,2027,low,72.50,40.00,48.707067
4,2027,medium,91.50,49.41,65.502915
5,2027,high,104.25,60.00,77.753074
6,2028,low,75.00,40.00,57.414135
7,2028,medium,103.00,49.41,81.595829
8,2028,high,118.50,60.00,95.506147
9,2029,low,77.50,40.00,66.121202


## 4. Maritime carbon cost per voyage

Run across Gayu's own scenario dimensions rather than collapsed to a single base case: three speed
scenarios for the gas carrier, both routings for the UK corridor, and both vessel sets.

In [5]:
maritime = runner.run_maritime_matrix()
print(f"{len(maritime)} voyage scenarios.")
outputs.maritime_summary(maritime)

312 voyage scenarios.


,corridor,vessel_set,year,distance_nm,voyage_days,voyage_co2_t,eu_ets_cost_eur,fueleu_cost_eur,uk_ets_cost_gbp,total_eur,total_gbp
92,halifax_hamburg,ACL G4-class ConRo,2026,2962,6.9,1593.8,64744.00,19518.977974,0.0000,84262.977974,0.0000
32,halifax_hamburg,VLGC/VLAC,2026,2962,8.3,1080.2,43880.00,13228.757709,0.0000,57108.757709,0.0000
265,ningbo_felixstowe,HMM Algeciras-class / Megamax-24,2026,10403,19.4,12464.4,0.00,0.000000,14510.7288,0.000000,14510.7288
169,ningbo_felixstowe,VLGC/VLAC,2026,10403,29.3,3813.0,0.00,0.000000,2935.9422,0.000000,2935.9422
98,halifax_hamburg,ACL G4-class ConRo,2027,2962,6.9,1593.8,74050.95,19518.977974,0.0000,93569.927974,0.0000
38,halifax_hamburg,VLGC/VLAC,2027,2962,8.3,1080.2,50187.75,13228.757709,0.0000,63416.507709,0.0000
268,ningbo_felixstowe,HMM Algeciras-class / Megamax-24,2027,10403,19.4,12464.4,0.00,0.000000,14510.7288,0.000000,14510.7288
172,ningbo_felixstowe,VLGC/VLAC,2027,10403,29.3,3813.0,0.00,0.000000,2935.9422,0.000000,2935.9422
104,halifax_hamburg,ACL G4-class ConRo,2028,2962,6.9,1593.8,83357.90,19518.977974,0.0000,102876.877974,0.0000
44,halifax_hamburg,VLGC/VLAC,2028,2962,8.3,1080.2,56495.50,13228.757709,0.0000,69724.257709,0.0000


### The asymmetry, without needing cargo tonnage

Dividing what each corridor pays by what it actually emits gives a like-for-like comparison that does not
depend on any missing input. The corridors remain in their own currencies.

This is the central finding stated as a number. Halifax–Hamburg is the shorter route and emits roughly a
third as much CO2, yet it pays around seventy times more per tonne emitted, because it sits inside a live
regulatory regime and the other does not.

In [6]:
eff = outputs.carbon_cost_per_tonne_co2(maritime)
eff[(eff['year'] == 2026) & (eff['price_scenario'] == 'medium')
    & (eff['speed_scenario'].isin(['base', 'service']))
    & (eff['route_scenario'] == 'suez')
    & (eff['uk_ets_variant'].isin(['n/a', 'current_scope']))][
    ['corridor', 'vessel_set', 'voyage_co2_t', 'currency',
     'cost_in_own_currency', 'effective_cost_per_tonne_co2']]

,corridor,vessel_set,voyage_co2_t,currency,cost_in_own_currency,effective_cost_per_tonne_co2
32,halifax_hamburg,VLGC/VLAC,1080.2,EUR,57108.757709,52.868689
33,halifax_hamburg,VLGC/VLAC,1080.2,EUR,43880.000000,40.622107
92,halifax_hamburg,ACL G4-class ConRo,1593.8,EUR,84262.977974,52.869229
93,halifax_hamburg,ACL G4-class ConRo,1593.8,EUR,64744.000000,40.622412
169,ningbo_felixstowe,VLGC/VLAC,3813.0,GBP,2935.942200,0.769982
265,ningbo_felixstowe,HMM Algeciras-class / Megamax-24,12464.4,GBP,14510.728800,1.164174


### The Red Sea diversion costs nothing under current UK rules

Rerouting Ningbo–Felixstowe via the Cape of Good Hope adds 4,412 nm and raises voyage emissions by about
42%, and changes the UK carbon bill by exactly zero, because UK ETS does not price the ocean leg. That is
a concrete illustration of how narrow the current UK scope is.

In [7]:
uk = maritime[(maritime['corridor'] == rc.NINGBO_FELIXSTOWE)
              & (maritime['year'] == 2026) & (maritime['price_scenario'] == 'medium')
              & (maritime['speed_scenario'] == 'base')
              & (maritime['vessel_set'] == 'VLGC/VLAC')]
uk[['route_scenario', 'distance_nm', 'voyage_co2_t', 'port_co2_t', 'uk_ets_cost_gbp']]

,route_scenario,distance_nm,voyage_co2_t,port_co2_t,uk_ets_cost_gbp
169,suez,10403,3813.0,58.51,2935.9422
193,cape,14815,5426.7,58.51,2935.9422


## 5. CBAM liability per tonne of product

Separate layer, separate unit. Still running on placeholder embedded emissions pending Riya's table.

One regulatory item remains unresolved: whether UK CBAM applies its own phase-in factor from 2027,
analogous to the EU schedule that starts at 2.5% in 2026, or charges the full amount from day one. The gap
between those two readings is wide, so the affected cases are skipped rather than guessed.

In [8]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    cbam_results = runner.run_cbam_matrix(emissions)
    for w in caught:
        print(w.message)

outputs.cbam_summary(cbam_results)[
    ['year', 'corridor', 'product', 'pathway', 'embedded_emissions_tco2e_per_tonne',
     'currency', 'cbam_cost_in_own_currency']]

,year,corridor,product,pathway,embedded_emissions_tco2e_per_tonne,currency,cbam_cost_in_own_currency
151,2026,halifax_hamburg,ammonia,cbam_default,1.980,EUR,1.128748
136,2026,halifax_hamburg,ammonia,green_electrolysis,0.620,EUR,0.321315
121,2026,halifax_hamburg,ammonia,grey_smr,2.180,EUR,1.129785
31,2026,halifax_hamburg,hydrogen,blue_smr_ccs,2.020,EUR,1.046865
46,2026,halifax_hamburg,hydrogen,cbam_default,10.820,EUR,6.168211
...,...,...,...,...,...,...,...
193,2030,ningbo_felixstowe,ammonia,green_electrolysis,0.818,GBP,13.323414
103,2030,ningbo_felixstowe,hydrogen,blue_ccs,6.280,GBP,102.287334
118,2030,ningbo_felixstowe,hydrogen,cbam_default,26.640,GBP,433.906780
88,2030,ningbo_felixstowe,hydrogen,coal_gasification,20.090,GBP,327.221742


## 6. Joining the layers: compliance cost per tonne

Gayu's cargo capacity notebook closes the gap between the two units. An 84,000 m³ carrier at the IGC Code
98% filling limit gives 82,320 m³ usable, which is **56,142 tonnes of ammonia or 5,828 tonnes of liquid
hydrogen**, a ratio of 9.6 to 1.

Every input is sourced: capacity from the same peer-reviewed vessel study already used for speed and port
time, the filling limit from IMO regulation, ammonia density from the NIH chemistry database, and hydrogen
density corroborated by a second peer-reviewed paper.

One caveat to carry into the write-up. An 84,000 m³ ammonia carrier runs at about -33°C and cannot
physically hold liquid hydrogen, which needs -253°C. The largest liquid hydrogen vessel built is 1,250 m³
and designs under development are around 40,000 m³. Applying the ammonia carrier's geometry to hydrogen
is a deliberate counterfactual that isolates the effect of cargo density by holding the vessel constant.
It should be labelled as such rather than presented as a shipping option available today.

In [9]:
print(f"Usable volume:  {vl.USABLE_VOLUME_M3:,.0f} m3 "
      f"({vl.VESSEL_CUBIC_CAPACITY_M3:,} x {vl.FILLING_LIMIT_FRACTION:.0%} IGC filling limit)")
for product, tonnes in vl.CARGO_TONNES.items():
    print(f"{product:<9} {tonnes:>7,} t per voyage  "
          f"(at {vl.DENSITY_KG_PER_M3[product]} kg/m3)")

with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter('always')
    compliance = runner.run_compliance_matrix(emissions)
    for w in caught:
        print(w.message)

Usable volume:  82,320 m3 (84,000 x 98% IGC filling limit)
ammonia    56,142 t per voyage  (at 682.0 kg/m3)
hydrogen    5,828 t per voyage  (at 70.8 kg/m3)


### Total compliance cost per tonne, 2026

This is the dissertation's central question answered in its proper unit. Currencies stay separate.

In [10]:
v = compliance[(compliance['year'] == 2026) & (compliance['price_scenario'] == 'medium')]
v[['corridor', 'product', 'pathway', 'currency', 'cbam_cost_per_tonne',
   'eu_ets_cost_per_tonne', 'fueleu_cost_per_tonne', 'uk_ets_cost_per_tonne',
   'total_compliance_cost_per_tonne']]

,corridor,product,pathway,currency,cbam_cost_per_tonne,eu_ets_cost_per_tonne,fueleu_cost_per_tonne,uk_ets_cost_per_tonne,total_compliance_cost_per_tonne
7,halifax_hamburg,hydrogen,green_electrolysis,EUR,0.637447,7.52917,2.269862,0.000000,10.436479
8,halifax_hamburg,hydrogen,grey_smr,EUR,5.218777,7.52917,2.269862,0.000000,15.017809
9,halifax_hamburg,hydrogen,blue_smr_ccs,EUR,1.046865,7.52917,2.269862,0.000000,10.845897
10,halifax_hamburg,hydrogen,cbam_default,EUR,6.168211,7.52917,2.269862,0.000000,15.967243
11,halifax_hamburg,ammonia,grey_smr,EUR,1.129785,0.78159,0.235630,0.000000,2.147005
12,halifax_hamburg,ammonia,green_electrolysis,EUR,0.321315,0.78159,0.235630,0.000000,1.338535
13,halifax_hamburg,ammonia,cbam_default,EUR,1.128748,0.78159,0.235630,0.000000,2.145968
112,ningbo_felixstowe,hydrogen,green_electrolysis,GBP,0.000000,0.00000,0.000000,0.503765,0.503765
113,ningbo_felixstowe,hydrogen,coal_gasification,GBP,0.000000,0.00000,0.000000,0.503765,0.503765
114,ningbo_felixstowe,hydrogen,blue_ccs,GBP,0.000000,0.00000,0.000000,0.503765,0.503765


### What 2026 shows

Halifax–Hamburg grey hydrogen pays about €18.35 per tonne in total carbon compliance. Ningbo–Felixstowe
hydrogen from coal gasification, at nearly twice the embedded emissions, pays about £0.50.

The dirtier product on the longer route pays roughly thirty-seven times less, purely because UK CBAM has
not started and UK ETS does not price the ocean leg. That is the regulatory asymmetry stated per tonne.

### And what 2030 shows

Run with UK CBAM assumed at 100%, which is a labelled what-if since the phase-in factor is unresolved, the
picture inverts. CBAM grows to dominate everything and the maritime terms become close to irrelevant.
Chinese coal-gasification hydrogen becomes the most exposed product in the study.

The asymmetry is therefore a window, not a permanent feature. Worth saying plainly in the discussion.

In [11]:
whatif = runner.run_compliance_matrix(
    emissions, uk_cbam_rate_override=1.0, skip_unresolved=False)
w2030 = whatif[(whatif['year'] == 2030) & (whatif['price_scenario'] == 'medium')
               & (whatif['uk_ets_variant'].isin(['n/a', 'current_scope']))]
w2030[['corridor', 'product', 'pathway', 'currency',
       'embedded_emissions_tco2e_per_tonne', 'cbam_cost_per_tonne',
       'maritime_cost_per_tonne', 'total_compliance_cost_per_tonne']]

,corridor,product,pathway,currency,embedded_emissions_tco2e_per_tonne,cbam_cost_per_tonne,maritime_cost_per_tonne,total_compliance_cost_per_tonne
91,halifax_hamburg,hydrogen,green_electrolysis,EUR,1.230,32.362838,19.802338,52.165176
92,halifax_hamburg,hydrogen,grey_smr,EUR,10.070,264.954287,19.802338,284.756626
93,halifax_hamburg,hydrogen,blue_smr_ccs,EUR,2.020,53.148725,19.802338,72.951063
94,halifax_hamburg,hydrogen,cbam_default,EUR,10.820,370.094043,19.802338,389.896381
95,halifax_hamburg,ammonia,grey_smr,EUR,2.180,57.358525,2.055645,59.414170
96,halifax_hamburg,ammonia,green_electrolysis,EUR,0.620,16.312975,2.055645,18.368620
97,halifax_hamburg,ammonia,cbam_default,EUR,1.980,67.725157,2.055645,69.780803
245,ningbo_felixstowe,hydrogen,green_electrolysis,GBP,2.340,115.619400,0.503765,116.123165
246,ningbo_felixstowe,hydrogen,coal_gasification,GBP,20.090,992.646900,0.503765,993.150665
247,ningbo_felixstowe,hydrogen,blue_ccs,GBP,6.280,310.294800,0.503765,310.798565


### Delivered cost is still blocked

The blocker has moved, though. Cargo tonnage has landed. What is missing now is the commercial side.

In [12]:
try:
    runner.run_delivered_cost()
except Exception as exc:
    print(exc)



  Delivered cost is still blocked, but the blocker has moved.

  RESOLVED: cargo tonnage, from Gayu's cargo_capacity_and_density_v2.ipynb.
    56,142 t ammonia, 5,828 t hydrogen. Compliance cost per tonne now runs,
    via run_compliance_matrix().

  STILL MISSING: production, conversion and freight cost per tonne.
    No student owns these in the data contracts. They are three of the six
    terms in a delivered cost, and production cost is the largest single
    component. See data/README.md.



## 7. Which assumption actually drives the maritime result

Each input varied by plus and minus 20% with everything else held constant. Run on the maritime layer only,
since that is the layer built entirely from sourced data.

Two results worth carrying into the write-up:

**FuelEU intensity dominates the EU corridor**, and by a lot more than 20%. The penalty is zero below the
threshold and rises steeply above it, so the response is strongly non-linear and a modest change in assumed
fuel quality swings the cost far more than a modest change in carbon price does.

**Port time is the only lever that matters on the UK corridor.** Speed and fuel intensity move it by
nothing at all, because neither affects time at berth, which is the only thing UK ETS prices.

In [13]:
sweep = pd.concat([sensitivity.sweep_corridor(c) for c in rc.CORRIDORS], ignore_index=True)
ranked = sensitivity.rank_drivers(sweep)

(ranked.groupby(['corridor', 'parameter'])['mean_abs_pct_change']
       .mean().unstack(0).round(2)
       .sort_values('halifax_hamburg', ascending=False))

corridor,halifax_hamburg,ningbo_felixstowe
parameter,,
fueleu_actual_intensity_gco2e_mj,129.70,0.00
service_speed_knots,21.08,0.00
engine_load_fraction,20.10,20.12
main_engine_power_kw,20.10,20.12
sfoc_g_per_kwh,20.10,20.12
carbon_price,20.00,20.00
vlsfo_carbon_factor,15.13,19.70
port_days,0.00,19.98


## 8. External cross-check

Ramsook, Boodlal and Maharaj (2025) put Trinidad and Tobago's ammonia CBAM burden at roughly 22% of export
revenue by 2034. The paper has since been read directly, which settled two questions that were open when
this check was first written. The burden is measured against EU-bound revenue only, not total export
revenue. And reproducing their figure needs the benchmark form of the CBAM obligation,
`max(0, embedded - benchmark x (1 - CBAM_factor))`, which lands at 20.7% against their published 22%.

The form this model currently uses, `embedded x CBAM_factor`, gives 14.5% on the same inputs. Both sit side
by side in `validation/reference_case.py` with a test that fails if the main one is switched without anyone
noticing. Switching is an open decision, waiting on the revised 2026-2030 EU ETS benchmarks adopted on
29 June 2026.

In [14]:
print(reference_case.format_reference_check(reference_case.run_reference_check()))

Reference case, Ramsook et al. (2025), T&T ammonia 2026-2034 average
  STATUS: calibrated, all inputs read from the paper.
  Burden is measured against EU-bound export revenue only.

  Published:                USD 115.0/t (22.0% of export value)

  This model:               USD 77.6/t (14.5%)  gap 7.5 pp  DIVERGES
  Benchmark mechanism:      USD 110.8/t (20.7%)  gap 1.3 pp  AGREES

  This model understates the published figure by a factor of 1.43.

  The two differ structurally, not numerically:
    this model    chargeable = embedded x CBAM_factor
    the paper     chargeable = embedded - benchmark x (1 - CBAM_factor)
  Only the second matches Regulation (EU) 2023/956 Article 31, under
  which free allocation shields a product benchmark rather than a
  share of the importer's own emissions. See the module docstring.



## 8b. Choice and timing

Not an optimisation - a ranking over the small set of pathways/corridors the literature actually supports.
Answers "which one, and when," not "what does it cost."

In [15]:
ranking = outputs.pathway_cost_ranking(emissions, commercial)
ranking[ranking.is_cheapest][['corridor', 'product', 'pathway', 'pathway_visible_cost_eur_per_tonne']]

,corridor,product,pathway,pathway_visible_cost_eur_per_tonne
0,halifax_hamburg,ammonia,grey_smr,504.16
2,halifax_hamburg,hydrogen,grey_smr,879.45
5,ningbo_felixstowe,ammonia,coal_gasification,533.91
7,ningbo_felixstowe,hydrogen,coal_gasification,1564.23


Cheapest pathway is production cost + CBAM only (conversion/shipping/maritime are pathway-invariant so they
cancel out of the ranking - see the function docstring for the caveat on why that cancellation could break).
At 2030 medium prices the dirtiest route wins everywhere, and stays cheapest under low/medium/high prices too
(`pathway_choice_price_robustness` below) - CBAM does not change the commercial choice.

In [16]:
outputs.pathway_choice_price_robustness(emissions, commercial)

,corridor,product,year,uk_price_variant,cheapest_pathway_low,cheapest_pathway_medium,cheapest_pathway_high,choice_stable,distinct_choices
0,halifax_hamburg,ammonia,2030,frozen,grey_smr,grey_smr,grey_smr,True,1
1,halifax_hamburg,hydrogen,2030,frozen,grey_smr,grey_smr,grey_smr,True,1
2,ningbo_felixstowe,ammonia,2030,frozen,coal_gasification,coal_gasification,coal_gasification,True,1
3,ningbo_felixstowe,hydrogen,2030,frozen,coal_gasification,coal_gasification,coal_gasification,True,1


In [17]:
outputs.corridor_crossover_year(compliance)

,product,pathway,price_scenario,first_year,last_year,cheaper_corridor_first_year,cheaper_corridor_last_year,cost_gap_first_year_gbp,cost_gap_last_year_gbp,crossover_year,ordering_changes
0,ammonia,cbam_default,medium,2026,2030,ningbo_felixstowe,halifax_hamburg,-1.78,11.53,2027,True
1,hydrogen,cbam_default,medium,2026,2030,ningbo_felixstowe,halifax_hamburg,-13.12,101.76,2027,True


Ningbo-Felixstowe is cheaper only in 2026, because UK CBAM does not exist yet. The ordering flips in 2027,
the year it starts, then the gap narrows as the EU's CBAM factor keeps ramping - one flip, not a slow overtake.

In [18]:
outputs.abatement_breakeven_year(emissions, commercial)

,corridor,product,pathway,reference_pathway,price_scenario,uk_price_variant,abatement_cost_eur_per_tco2,carbon_price_first_year,carbon_price_last_year,carbon_price_varies_by_year,verdict_first_year,verdict_last_year,breakeven_year,first_marginal_year,note
0,halifax_hamburg,ammonia,green_electrolysis,grey_smr,medium,frozen,308.33,80.00,126.00,True,not justified,not justified,NaN,NaN,
1,halifax_hamburg,hydrogen,blue_smr_ccs,grey_smr,medium,frozen,54.52,80.00,126.00,True,justified,justified,2026.0,NaN,
2,halifax_hamburg,hydrogen,green_electrolysis,grey_smr,medium,frozen,338.61,80.00,126.00,True,not justified,not justified,NaN,NaN,
3,ningbo_felixstowe,ammonia,green_electrolysis,coal_gasification,medium,frozen,57.31,57.91,57.91,False,marginal,marginal,NaN,2026.0,Carbon price is flat across the horizon because the UK ETS price is frozen at the sourced 2026 determination. The verdict cannot change by construction. Not evidence about timing.
4,ningbo_felixstowe,hydrogen,blue_ccs,coal_gasification,medium,frozen,40.04,57.91,57.91,False,justified,justified,2026.0,NaN,Carbon price is flat across the horizon because the UK ETS price is frozen at the sourced 2026 determination. The verdict cannot change by construction. Not evidence about timing.
5,ningbo_felixstowe,hydrogen,green_electrolysis,coal_gasification,medium,frozen,238.61,57.91,57.91,False,not justified,not justified,NaN,NaN,Carbon price is flat across the horizon because the UK ETS price is frozen at the sourced 2026 determination. The verdict cannot change by construction. Not evidence about timing.


Check `carbon_price_varies_by_year` before reading a UK row: the UK ETS price is frozen (only 2026 was ever
sourced), so a UK pathway showing no breakeven year is an artefact of that, not evidence switching never pays.

## 9. Outputs

In [19]:
written = outputs.write_all(maritime, cbam_results, sweep, ranked, compliance,
                            emissions=emissions, commercial=commercial)
for name in written:
    print(f"cbam_model/outputs/{name}")

cbam_model/outputs/bunker_fuel_comparison.csv
cbam_model/outputs/marginal_abatement_cost.csv
cbam_model/outputs/abatement_source_robustness.csv
cbam_model/outputs/pathway_cost_ranking.csv
cbam_model/outputs/pathway_choice_price_robustness.csv
cbam_model/outputs/abatement_breakeven_year.csv
cbam_model/outputs/cbam_mechanism_comparison.csv
cbam_model/outputs/compliance_cost_per_tonne.csv
cbam_model/outputs/corridor_cost_comparison.csv
cbam_model/outputs/corridor_crossover_year.csv
cbam_model/outputs/corridor_lock_in.csv
cbam_model/outputs/switching_cost_sensitivity.csv
cbam_model/outputs/competitiveness_burden.csv
cbam_model/outputs/competitiveness_asymmetry.csv
cbam_model/outputs/maritime_cost_per_voyage.csv
cbam_model/outputs/maritime_summary.csv
cbam_model/outputs/effective_carbon_cost.csv
cbam_model/outputs/cbam_cost_per_tonne.csv
cbam_model/outputs/sensitivity_sweep.csv
cbam_model/outputs/sensitivity_ranked.csv
cbam_model/outputs/compliance_sensitivity_sweep.csv
cbam_model/outputs/c

## Where this stands

**Solid.** The maritime layer is complete and carries no invented inputs. All 25 of Gayu's published
figures reproduce exactly. The UK ETS price anchor is resolved from an official determination. The
regulatory logic is pinned by 72 tests, including the three facts this project has previously got wrong.

**Resolved since the first build.** UK ETS price anchors, from the official 2026 determination. Cargo
tonnage, from Gayu's capacity notebook. Compliance cost per tonne now runs end to end.

**Blocked on other people.**

1. **Embedded emissions**, from Riya. The CBAM layer runs on placeholders, and by 2030 CBAM dominates every
   other term, so this is now the largest source of uncertainty in the study. The origin carbon price
   column matters most: Canada prices industrial carbon and China prices it lower, and a large enough
   Canadian price would cut Halifax-Hamburg's CBAM liability sharply.
2. **Production, conversion and freight cost per tonne.** No owner assigned. The only thing standing
   between compliance cost and full delivered cost.
3. **The UK CBAM phase-in factor.** Blocks the 2027-onward UK cases and drives the entire 2030 comparison.
4. **The Ramsook cross-check**, which does not reconcile and needs the paper read.